# TP 0 — Your first regression model with PyTorch

**IMT · Introduction to Machine Learning · Google Colab**

Before binary classification, let's start with something simpler: **predict a number**.

### Scenario: estimating a taxi fare from a trip's distance

We will create a small **synthetic dataset**: longer trips usually cost more, with some random variation. These are invented teaching examples, not real taxi prices.

**Goals:** understand features and targets; train a simple linear model; plot its predictions and loss; evaluate its numerical error.

**How to use:** Run the prepared cells in order. All code is provided for the first demonstration. Then try two very small experiments. No GPU or API key required.

**Expected time:** 15–20 minutes.

## 1. Generate and visualize the data

Each example has one **feature**: distance in kilometers, and one **target**: fare in euros.

We generate 120 examples using a simple line plus random noise. Don't worry about the code that generates the fake data; focus on the scatter plot.

**Question:** Is this regression or classification? Why?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# Synthetic teaching data (not real taxi prices).
distance = rng.uniform(0.5, 12.0, size=120)
fare = 4.0 + 1.8 * distance + rng.normal(0, 1.1, size=len(distance))

X = distance.reshape(-1, 1)  # Feature: distance
y = fare.reshape(-1, 1)      # Target: fare

plt.scatter(X[:, 0], y[:, 0], s=20, alpha=0.8)
plt.xlabel('Distance (km)')
plt.ylabel('Fare (€)')
plt.title('Synthetic taxi trips')
plt.show()
print('X shape:', X.shape, '| y shape:', y.shape)

## 2. Split training and test data

We will use **80% of the trips to train** and the other **20% to test**. The network never uses test labels to update its parameters.

PyTorch uses **tensors** as inputs and outputs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)
xte = torch.tensor(X_test, dtype=torch.float32)

print(f'Train: {len(xtr)} trips | Test: {len(xte)} trips')

## 3. Define the simplest neural network

With **one input and one output**, nn.Linear(1, 1) learns a straight line. Its parameters are a **weight (slope)** and a **bias (intercept)**.

For a continuous quantity such as euros, we use **mean squared error (MSE)**. We use **Adam** to adjust the parameters.

**Question:** Why do we use MSE rather than binary cross-entropy (the classification loss)?

In [ ]:
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print('Model:', model)
print('Loss:', criterion)
print('Optimizer: Adam, learning rate = 0.05')

## 4. Train the model

Follow these **five steps** inside the loop:

1. Clear previous gradients.
2. Compute predicted fares.
3. Compute the loss against actual fares.
4. Backpropagate.
5. Update the model parameters.

Because the dataset is small, we use the whole training set in each step. The graph shows how the error decreases.

In [ ]:
history = []
for epoch in range(300):
    optimizer.zero_grad()                  # 1. Clear gradients
    predicted_fares = model(xtr)           # 2. Forward pass
    loss = criterion(predicted_fares, ytr) # 3. MSE loss
    loss.backward()                        # 4. Backward pass
    optimizer.step()                       # 5. Parameter update
    history.append(loss.item())

plt.plot(history)
plt.xlabel('Epoch')
plt.ylabel('Training MSE (€²)')
plt.title('The loss decreases as the model learns')
plt.show()
print(f'Initial loss: {history[0]:.2f} | Final loss: {history[-1]:.2f}')

## 5. Evaluate on test trips

Regression predictions are **numbers**, not classes. We use **mean absolute error (MAE)**: the average absolute distance between prediction and true fare, measured in euros.

The chart compares the observed trips with the straight line learned by our model.

In [ ]:
model.eval()
with torch.no_grad():
    test_predictions = model(xte).numpy().ravel()

mae = mean_absolute_error(y_test.ravel(), test_predictions)
print(f'Test MAE: €{mae:.2f}')

order = np.argsort(X_test[:, 0])
plt.scatter(X_train[:, 0], y_train[:, 0], s=18, alpha=0.45, label='Training data')
plt.scatter(X_test[:, 0], y_test[:, 0], s=35, label='Test data')
plt.plot(X_test[order, 0], test_predictions[order], linewidth=2.5, label='Learned line')
plt.xlabel('Distance (km)')
plt.ylabel('Fare (€)')
plt.legend()
plt.title('Regression: a continuous prediction')
plt.show()

## 6. Predict the fare for a new trip

Give the model a new trip distance. It outputs an **estimated fare**, rather than a 0/1 label.

The final two numbers show the **learned slope** and **intercept**.

In [ ]:
new_distance = 7.0  # Try 2.0 or 10.0 too.
with torch.no_grad():
    estimated_fare = model(torch.tensor([[new_distance]], dtype=torch.float32)).item()
print(f'Estimated fare for {new_distance:.1f} km: €{estimated_fare:.2f}')

weight = model.weight.item()
bias = model.bias.item()
print(f'Learned relationship: fare ≈ {weight:.2f} × distance + {bias:.2f}')

## 7. Your turn — two easy experiments

**Experiment A:** Change new_distance from 7.0 to 2.0 and then to 10.0 in Section 6. How does the prediction change?

**Experiment B:** Change range(300) to range(30) in Section 4. **Re-run Sections 3–6** so training starts with a fresh model. Does the loss fall as much?

### Three short questions

1. What do X and y represent?
2. Why use MSE rather than binary cross-entropy?
3. Why evaluate on trips not seen during training?

**Takeaway:** Regression predicts a **continuous numerical value**, while classification predicts a **category**. The training loop is similar, but the output meaning and loss function are different.